<h2 style="
background-color:#F8D7DA;
color:black;
padding:10px;
border-radius:8px;
font-weight:bold;
font-style:italic;
">
Task 4 : Financial Risk Identification
</h2>

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [7]:
df = pd.read_csv("cleaned_jp_morgan.csv")

In [8]:
df.head()

,TransactionID,CustomerID,AccountID,AccountType,TransactionType,Product,Firm,Region,Manager,TransactionDate,TransactionAmount,AccountBalance,RiskScore,CreditRating,TenureMonths,Year,Month
0,3,CUST2412,ACC80131,loan,withdrawal,Personal Loan,Firm C,West,Manager 3,2023-08-06,33759.69057,126486.40830,0.225824,611,89,2023,8
1,32,CUST1467,ACC74631,current,withdrawal,Home Loan,Firm D,North,Manager 2,2023-11-08,69319.19933,24834.76291,0.335717,817,174,2023,11
2,9,CUST2699,ACC39482,loan,withdrawal,Credit Card,Firm D,West,Manager 4,2024-05-15,42831.48483,123007.43530,0.572453,332,31,2024,5
3,42,CUST9535,ACC82947,current,withdrawal,Home Loan,Firm A,South,Manager 4,2023-04-30,70903.79697,73073.64225,0.571993,626,92,2023,4
4,166,CUST7459,ACC39500,credit,payment,Home Loan,Firm D,South,Manager 4,2023-02-16,21948.97355,113405.32820,0.380675,411,13,2023,2


**<h3 style="color:blue;">4.1 Track Accounts with Frequent Large Withdrawals.</h3>**

##### Frequent Large Withdrawals
##### Rubric:
##### Large Withdrawal = Top 10% Withdrawal Amounts

In [11]:

withdrawals = df[df["TransactionType"] == "withdrawal"].copy()

large_limit = withdrawals["TransactionAmount"].quantile(0.90)

large_withdrawals = withdrawals[
    withdrawals["TransactionAmount"] >= large_limit
]

withdrawal_summary = (
    large_withdrawals
    .groupby("AccountID")
    .size()
    .reset_index(name="Large_Withdrawal_Count")
)

withdrawal_summary.sort_values(
    by="Large_Withdrawal_Count",
    ascending=False
)

,AccountID,Large_Withdrawal_Count
15,ACC81631,2
0,ACC15925,1
11,ACC67701,1
19,ACC99117,1
18,ACC88252,1
17,ACC86784,1
16,ACC82926,1
14,ACC77592,1
13,ACC76549,1
12,ACC74656,1


**<h3 style="color:blue;">4.2 Detect Possible Overdraft Accounts.</h>**

#### Accounts with Negative Balance

In [14]:
overdraft_accounts = df[
    df["AccountBalance"] < 0
]

overdraft_accounts[
    ["AccountID",
     "CustomerID",
     "AccountBalance"]
].drop_duplicates()

,AccountID,CustomerID,AccountBalance
102,ACC57516,CUST4769,-1308.418354
121,ACC50817,CUST9535,-13812.693060
147,ACC49774,CUST6028,-584.724017
482,ACC69323,CUST6947,-8873.804269
543,ACC61926,CUST8461,-9976.162775
568,ACC30852,CUST7395,-13018.400900
614,ACC77638,CUST8318,-4782.075751
621,ACC57597,CUST9525,-12919.394960
733,ACC57700,CUST4373,-1164.041546
756,ACC45907,CUST2915,-3430.668059


**<h3 style="color:blue;">4.3 Calculate Balance Volatility.</h>**

#### Balance Volatility

In [17]:

balance_volatility = (
    df.groupby("AccountID")["AccountBalance"]
      .std()
      .reset_index(name="Balance_STD")
)

balance_volatility = balance_volatility.sort_values(
    by="Balance_STD",
    ascending=False
)

balance_volatility.head(10)

,AccountID,Balance_STD
73,ACC39529,87925.613732
155,ACC78589,74986.233658
98,ACC49774,67777.809369
45,ACC28612,59940.470436
112,ACC57516,57794.003514
104,ACC51971,56709.662822
60,ACC33287,55392.907529
142,ACC74656,55037.331771
129,ACC66190,52142.885192
176,ACC90887,51988.111482


**<h3 style="color:blue;">4.4 Coefficient of Variation.</h>**

#### Coefficient of Variation

In [20]:
balance_stats = (
    df.groupby("AccountID")["AccountBalance"]
      .agg(["mean","std"])
      .reset_index()
)

balance_stats["CV"] = (
    balance_stats["std"] /
    balance_stats["mean"]
).abs()

balance_stats.sort_values(
    by="CV",
    ascending=False
).head(10)

,AccountID,mean,std,CV
98,ACC49774,47341.424602,67777.809369,1.431681
73,ACC39529,75349.812090,87925.613732,1.166899
155,ACC78589,70449.551685,74986.233658,1.064396
119,ACC61827,46163.402105,48703.937356,1.055034
150,ACC77638,47234.380597,46413.053541,0.982612
43,ACC28295,50616.389573,48781.063965,0.963740
109,ACC54589,42538.305527,40650.871773,0.955630
112,ACC57516,60675.510231,57794.003514,0.952510
84,ACC45521,46750.408865,44369.532622,0.949073
115,ACC57872,50980.576460,47910.887480,0.939787


**<h3 style="color:blue;">4.5 Detect Anomalies Using IQR.</h>**

#### IQR Method

In [23]:
Q1 = df["TransactionAmount"].quantile(0.25)
Q3 = df["TransactionAmount"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

anomalies = df[
    (df["TransactionAmount"] < lower_limit) |
    (df["TransactionAmount"] > upper_limit)
]

anomalies

,TransactionID,CustomerID,AccountID,AccountType,TransactionType,Product,Firm,Region,Manager,TransactionDate,TransactionAmount,AccountBalance,RiskScore,CreditRating,TenureMonths,Year,Month
53,178,CUST1189,ACC50439,savings,deposit,Savings Account,Firm D,East,Manager 1,2024-06-16,-60979.07348,57916.06101,0.160988,655,17,2024,6
97,67,CUST6391,ACC26940,current,payment,Home Loan,Firm A,South,Manager 4,2023-03-16,-25964.94612,94822.29734,0.639771,315,209,2023,3
115,122,CUST8344,ACC99117,credit,withdrawal,Home Loan,Firm B,East,Manager 4,2024-01-29,129471.46560,73117.85560,0.186813,820,210,2024,1
129,161,CUST4373,ACC41829,current,payment,Savings Account,Firm A,North,Manager 3,2024-03-22,135972.34390,42302.00951,-0.368537,469,175,2024,3
242,86,CUST7855,ACC95164,savings,payment,Home Loan,Firm C,North,Manager 3,2023-10-26,143067.18460,73627.79044,0.752617,582,226,2023,10
397,51,CUST1497,ACC88074,current,transfer,Mutual Fund,Firm B,East,Manager 3,2024-03-05,-45352.95439,96125.56424,-0.203847,668,36,2024,3
579,50,CUST1749,ACC99117,current,payment,Personal Loan,Firm E,West,Manager 4,2023-11-29,-24819.84559,101124.98330,0.967253,628,202,2023,11
659,21,CUST1644,ACC77533,credit,payment,Personal Loan,Firm D,South,Manager 4,2023-12-24,-30826.73980,68006.03943,0.488742,678,42,2023,12
701,191,CUST6837,ACC46655,current,deposit,Mutual Fund,Firm E,West,Manager 2,2024-06-16,-29563.97803,49810.32678,0.517913,466,150,2024,6
727,163,CUST1747,ACC92104,savings,deposit,Savings Account,Firm A,West,Manager 1,2023-08-06,147447.29510,101759.29850,0.099626,421,86,2023,8


**<h3 style="color:blue;"> 4.6 Detect Anomalies Using Z-Score.</h>**

In [25]:
from scipy.stats import zscore

##### Z-Score Method

In [27]:
df["Z_Score"] = zscore(df["TransactionAmount"])

zscore_outliers = df[
    df["Z_Score"].abs() > 3
]

zscore_outliers

,TransactionID,CustomerID,AccountID,AccountType,TransactionType,Product,Firm,Region,Manager,TransactionDate,TransactionAmount,AccountBalance,RiskScore,CreditRating,TenureMonths,Year,Month,Z_Score
53,178,CUST1189,ACC50439,savings,deposit,Savings Account,Firm D,East,Manager 1,2024-06-16,-60979.07348,57916.06101,0.160988,655,17,2024,6,-3.856562
242,86,CUST7855,ACC95164,savings,payment,Home Loan,Firm C,North,Manager 3,2023-10-26,143067.18460,73627.79044,0.752617,582,226,2023,10,3.165364
397,51,CUST1497,ACC88074,current,transfer,Mutual Fund,Firm B,East,Manager 3,2024-03-05,-45352.95439,96125.56424,-0.203847,668,36,2024,3,-3.318814
727,163,CUST1747,ACC92104,savings,deposit,Savings Account,Firm A,West,Manager 1,2023-08-06,147447.29510,101759.29850,0.099626,421,86,2023,8,3.316099


**<h3 style="color:blue;">4.7 Highlight Suspicious Customers.</h>**

##### Suspicious Customer Identification

In [30]:
suspicious_customers = pd.concat([
    large_withdrawals,
    overdraft_accounts,
    anomalies
])

suspicious_customers = suspicious_customers.drop_duplicates(
    subset=["CustomerID"]
)

suspicious_customers[
    [
        "CustomerID",
        "AccountID",
        "TransactionType",
        "TransactionAmount",
        "AccountBalance"
    ]
]

,CustomerID,AccountID,TransactionType,TransactionAmount,AccountBalance
9,CUST3041,ACC81631,withdrawal,105066.72200,122057.461700
80,CUST7388,ACC81631,withdrawal,98620.49477,35356.894290
115,CUST8344,ACC99117,withdrawal,129471.46560,73117.855600
130,CUST6776,ACC77592,withdrawal,97036.39944,55451.323600
134,CUST9248,ACC82926,withdrawal,93384.52157,60905.647180
138,CUST8279,ACC45907,withdrawal,125896.51730,26843.022250
158,CUST7395,ACC99409,withdrawal,97158.08018,54295.877660
175,CUST9962,ACC40952,withdrawal,114922.64320,78680.764190
248,CUST4763,ACC88252,withdrawal,90962.08527,50827.238940
322,CUST5255,ACC40939,withdrawal,105938.61820,77344.763910


**<h3 style="color:blue;">4.8 Summary.</h>**

##### Financial Risk Summary

In [33]:
print("=" * 70)
print("Financial Risk Identification Summary")
print("=" * 70)

print("Accounts with Large Withdrawals :", withdrawal_summary["AccountID"].nunique())
print("Overdraft Accounts              :", overdraft_accounts["AccountID"].nunique())
print("IQR Anomalies                   :", len(anomalies))
print("Z-Score Anomalies               :", len(zscore_outliers))
print("Suspicious Customers            :", suspicious_customers["CustomerID"].nunique())

Financial Risk Identification Summary
Accounts with Large Withdrawals : 21
Overdraft Accounts              : 10
IQR Anomalies                   : 10
Z-Score Anomalies               : 4
Suspicious Customers            : 33
